<a href="https://colab.research.google.com/github/ChrisEsau/ufc-ai-clv-tracker/blob/dev/UFC_rolling_dataset_V4_refactored.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# =========================
# MOUNT GOOGLE DRIVE
# =========================

from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [ ]:
# =========================
# PATH SETUP
# =========================

import os
import sys
import pandas as pd
import numpy as np
from collections import defaultdict, deque

sys.path.append("/content/drive/MyDrive/UFC_AI")
from ufc_pipeline_utils import *

paths = UFCPipelinePaths(
    base_path="/content/drive/MyDrive/UFC_AI",
    model_version="UFC_Model_v5_Experiment"
)

BASE_PATH = paths.base_path
RAW_CSV_PATH = f"{BASE_PATH}/UFC.csv"
ROLLING_CSV_PATH = f"{BASE_PATH}/UFC_enhanced_rolling_features.csv"

ensure_dir(BASE_PATH)

print("Base path:", BASE_PATH)

from ufc_feature_engineering import *

Base path: /content/drive/MyDrive/UFC_AI


In [ ]:
# =========================
# LOAD RAW UFC DATA
# =========================

df = pd.read_csv(
    "/content/drive/MyDrive/UFC_AI/UFC.csv"
)


df["date"] = pd.to_datetime(df["date"], errors="coerce")

df = df.sort_values("date").reset_index(drop=True)

df = df.dropna(
    subset=[
        "date",
        "r_id",
        "b_id",
        "winner_id"
    ]
)

df["target"] = (
    df["winner_id"] == df["r_id"]
).astype(int)

print(df.shape)
print(df.head())

(8190, 129)
           event_id         event_name       date               location  \
0  a6a9ab5a824e8f66  UFC 2: No Way Out 1994-03-11  Denver, Colorado, USA   
1  a6a9ab5a824e8f66  UFC 2: No Way Out 1994-03-11  Denver, Colorado, USA   
2  a6a9ab5a824e8f66  UFC 2: No Way Out 1994-03-11  Denver, Colorado, USA   
3  a6a9ab5a824e8f66  UFC 2: No Way Out 1994-03-11  Denver, Colorado, USA   
4  a6a9ab5a824e8f66  UFC 2: No Way Out 1994-03-11  Denver, Colorado, USA   

           fight_id      division  title_fight      method  finish_round  \
0  4acab67848e78327   open weight            0  Submission             1   
1  00835554f95fa911  2 tournament            1      KO/TKO             1   
2  aa161c7159741766   open weight            0  Submission             1   
3  5655639baecd8f6a   open weight            0  Submission             1   
4  8fbdde20b712b8da   open weight            0  Submission             1   

   match_time_sec  ...  b_td_avg_acc b_td_def b_sub_avg         winner  \


In [ ]:
# =========================
# SETTINGS + HELPERS
# =========================

START_ELO = 1500
K_FACTOR = 32
RECENT_N = 3

def safe_div(a, b):
    return a / b if b != 0 else 0

def expected_score(elo_a, elo_b):
    return 1 / (1 + 10 ** ((elo_b - elo_a) / 400))

In [ ]:
# =========================
# FIGHTER STATE
# =========================

def default_state():
    return {
        "elo": START_ELO,

        "fights": 0,
        "wins": 0,
        "losses": 0,

        "kd_for": 0,
        "kd_against": 0,

        "sig_str_landed": 0,
        "sig_str_attempted": 0,
        "sig_str_absorbed": 0,
        "sig_str_attempted_against": 0,

        "td_landed": 0,
        "td_attempted": 0,
        "td_allowed": 0,
        "td_attempted_against": 0,

        "sub_att": 0,
        "ctrl": 0,
        "ctrl_against": 0,

        "fight_time_sec": 0,

        "finish_wins": 0,
        "ko_wins": 0,
        "sub_wins": 0,
        "decision_wins": 0,

        "finish_losses": 0,
        "decision_losses": 0,

        "opponent_elo_sum": 0,
        "best_win_elo": START_ELO,
        "worst_loss_elo": START_ELO,

        "win_streak": 0,
        "loss_streak": 0,
        "last_fight_date": None,

        "recent_results": deque(maxlen=RECENT_N),
        "recent_sig_landed": deque(maxlen=RECENT_N),
        "recent_sig_absorbed": deque(maxlen=RECENT_N),
        "recent_td_landed": deque(maxlen=RECENT_N),
        "recent_finish_results": deque(maxlen=RECENT_N),
        "recent_fight_times": deque(maxlen=RECENT_N),
    }

fighter_state = defaultdict(default_state)

In [ ]:
# =========================
# PREFIGHT FEATURE FUNCTION
# =========================

def get_prefight_features(fighter_id, fight_date):

    s = fighter_state[fighter_id]

    minutes = s["fight_time_sec"] / 60
    fifteen_min_units = s["fight_time_sec"] / 900

    if s["last_fight_date"] is None:
        days_since_last_fight = 365
    else:
        days_since_last_fight = (
            fight_date - s["last_fight_date"]
        ).days

    return {
        "elo": s["elo"],

        "fights": s["fights"],
        "wins": s["wins"],
        "losses": s["losses"],
        "win_pct": safe_div(s["wins"], s["fights"]),

        "kd_avg": safe_div(s["kd_for"], s["fights"]),
        "kd_absorbed_avg": safe_div(s["kd_against"], s["fights"]),

        "splm": safe_div(s["sig_str_landed"], minutes),
        "sapm": safe_div(s["sig_str_absorbed"], minutes),

        "str_acc": safe_div(
            s["sig_str_landed"],
            s["sig_str_attempted"]
        ),

        "str_def": 1 - safe_div(
            s["sig_str_absorbed"],
            s["sig_str_attempted_against"]
        ),

        "td_avg": safe_div(s["td_landed"], fifteen_min_units),

        "td_acc": safe_div(
            s["td_landed"],
            s["td_attempted"]
        ),

        "td_def": 1 - safe_div(
            s["td_allowed"],
            s["td_attempted_against"]
        ),

        "sub_avg": safe_div(s["sub_att"], fifteen_min_units),

        "ctrl_per_min": safe_div(s["ctrl"], minutes),
        "ctrl_against_per_min": safe_div(s["ctrl_against"], minutes),

        "finish_rate": safe_div(s["finish_wins"], s["wins"]),
        "ko_rate": safe_div(s["ko_wins"], s["wins"]),
        "sub_win_rate": safe_div(s["sub_wins"], s["wins"]),
        "decision_win_rate": safe_div(s["decision_wins"], s["wins"]),

        "finish_loss_rate": safe_div(s["finish_losses"], s["losses"]),
        "decision_loss_rate": safe_div(s["decision_losses"], s["losses"]),

        "avg_opponent_elo": safe_div(
            s["opponent_elo_sum"],
            s["fights"]
        ),

        "best_win_elo": s["best_win_elo"],
        "worst_loss_elo": s["worst_loss_elo"],

        "avg_fight_time": safe_div(
            s["fight_time_sec"],
            s["fights"]
        ),

        "win_streak": s["win_streak"],
        "loss_streak": s["loss_streak"],
        "days_since_last_fight": days_since_last_fight,

        "recent_win_pct": safe_div(
            sum(s["recent_results"]),
            len(s["recent_results"])
        ),

        "recent_splm": safe_div(
            sum(s["recent_sig_landed"]),
            len(s["recent_sig_landed"])
        ),

        "recent_sapm": safe_div(
            sum(s["recent_sig_absorbed"]),
            len(s["recent_sig_absorbed"])
        ),

        "recent_td_avg": safe_div(
            sum(s["recent_td_landed"]),
            len(s["recent_td_landed"])
        ),

        "recent_finish_rate": safe_div(
            sum(s["recent_finish_results"]),
            len(s["recent_finish_results"])
        ),

        "recent_avg_fight_time": safe_div(
            sum(s["recent_fight_times"]),
            len(s["recent_fight_times"])
        ),
    }

In [ ]:
# =========================
# UPDATE FIGHTER FUNCTION
# =========================

def update_fighter(
    fighter_id,
    fight_date,
    won,
    method,
    own,
    opp,
    fight_time_sec,
    opponent_elo
):

    s = fighter_state[fighter_id]

    method = str(method)

    s["opponent_elo_sum"] += opponent_elo

    s["fights"] += 1

    if won:
        s["wins"] += 1
        s["win_streak"] += 1
        s["loss_streak"] = 0

        s["best_win_elo"] = max(
            s["best_win_elo"],
            opponent_elo
        )

        if method in ["KO/TKO", "TKO - Doctor's Stoppage"]:
            s["finish_wins"] += 1
            s["ko_wins"] += 1

        elif method == "Submission":
            s["finish_wins"] += 1
            s["sub_wins"] += 1

        elif "Decision" in method:
            s["decision_wins"] += 1

    else:
        s["losses"] += 1
        s["loss_streak"] += 1
        s["win_streak"] = 0

        s["worst_loss_elo"] = min(
            s["worst_loss_elo"],
            opponent_elo
        )

        if method in ["KO/TKO", "TKO - Doctor's Stoppage", "Submission"]:
            s["finish_losses"] += 1

        elif "Decision" in method:
            s["decision_losses"] += 1

    s["kd_for"] += own["kd"]
    s["kd_against"] += opp["kd"]

    s["sig_str_landed"] += own["sig_str_landed"]
    s["sig_str_attempted"] += own["sig_str_attempted"]
    s["sig_str_absorbed"] += opp["sig_str_landed"]
    s["sig_str_attempted_against"] += opp["sig_str_attempted"]

    s["td_landed"] += own["td_landed"]
    s["td_attempted"] += own["td_attempted"]
    s["td_allowed"] += opp["td_landed"]
    s["td_attempted_against"] += opp["td_attempted"]

    s["sub_att"] += own["sub_att"]

    s["ctrl"] += own["ctrl"]
    s["ctrl_against"] += opp["ctrl"]

    s["fight_time_sec"] += fight_time_sec
    s["last_fight_date"] = fight_date

    fight_minutes = fight_time_sec / 60

    finish_flag = 1 if method in [
        "KO/TKO",
        "TKO - Doctor's Stoppage",
        "Submission"
    ] else 0

    s["recent_results"].append(1 if won else 0)

    s["recent_sig_landed"].append(
        safe_div(own["sig_str_landed"], fight_minutes)
    )

    s["recent_sig_absorbed"].append(
        safe_div(opp["sig_str_landed"], fight_minutes)
    )

    s["recent_td_landed"].append(
        own["td_landed"]
    )

    s["recent_finish_results"].append(
        finish_flag
    )

    s["recent_fight_times"].append(
        fight_time_sec
    )

In [ ]:
# =========================
# BUILD ROLLING DATASET
# =========================

rolling_rows = []

for _, row in df.iterrows():

    fight_date = row["date"]
    fight_time_sec = row["match_time_sec"]

    r_id = row["r_id"]
    b_id = row["b_id"]

    r_pre = get_prefight_features(r_id, fight_date)
    b_pre = get_prefight_features(b_id, fight_date)

    new_row = row.to_dict()

    for key in r_pre:
        new_row[f"r_pre_{key}"] = r_pre[key]
        new_row[f"b_pre_{key}"] = b_pre[key]
        new_row[f"{key}_diff"] = r_pre[key] - b_pre[key]

    rolling_rows.append(new_row)

    r_elo = fighter_state[r_id]["elo"]
    b_elo = fighter_state[b_id]["elo"]

    r_expected = expected_score(r_elo, b_elo)
    b_expected = expected_score(b_elo, r_elo)

    r_actual = row["target"]
    b_actual = 1 - row["target"]

    fighter_state[r_id]["elo"] = (
        r_elo + K_FACTOR * (r_actual - r_expected)
    )

    fighter_state[b_id]["elo"] = (
        b_elo + K_FACTOR * (b_actual - b_expected)
    )

    r_stats = {
        "kd": row["r_kd"],
        "sig_str_landed": row["r_sig_str_landed"],
        "sig_str_attempted": row["r_sig_str_atmpted"],
        "td_landed": row["r_td_landed"],
        "td_attempted": row["r_td_atmpted"],
        "sub_att": row["r_sub_att"],
        "ctrl": row["r_ctrl"],
    }

    b_stats = {
        "kd": row["b_kd"],
        "sig_str_landed": row["b_sig_str_landed"],
        "sig_str_attempted": row["b_sig_str_atmpted"],
        "td_landed": row["b_td_landed"],
        "td_attempted": row["b_td_atmpted"],
        "sub_att": row["b_sub_att"],
        "ctrl": row["b_ctrl"],
    }

    update_fighter(
        r_id,
        fight_date,
        won=(row["target"] == 1),
        method=row["method"],
        own=r_stats,
        opp=b_stats,
        fight_time_sec=fight_time_sec,
        opponent_elo=b_elo
    )

    update_fighter(
        b_id,
        fight_date,
        won=(row["target"] == 0),
        method=row["method"],
        own=b_stats,
        opp=r_stats,
        fight_time_sec=fight_time_sec,
        opponent_elo=r_elo
    )

rolling_df = pd.DataFrame(rolling_rows)

print(rolling_df.shape)
print(rolling_df.head())

(8190, 237)
           event_id         event_name       date               location  \
0  a6a9ab5a824e8f66  UFC 2: No Way Out 1994-03-11  Denver, Colorado, USA   
1  a6a9ab5a824e8f66  UFC 2: No Way Out 1994-03-11  Denver, Colorado, USA   
2  a6a9ab5a824e8f66  UFC 2: No Way Out 1994-03-11  Denver, Colorado, USA   
3  a6a9ab5a824e8f66  UFC 2: No Way Out 1994-03-11  Denver, Colorado, USA   
4  a6a9ab5a824e8f66  UFC 2: No Way Out 1994-03-11  Denver, Colorado, USA   

           fight_id      division  title_fight      method  finish_round  \
0  4acab67848e78327   open weight            0  Submission             1   
1  00835554f95fa911  2 tournament            1      KO/TKO             1   
2  aa161c7159741766   open weight            0  Submission             1   
3  5655639baecd8f6a   open weight            0  Submission             1   
4  8fbdde20b712b8da   open weight            0  Submission             1   

   match_time_sec  ...  recent_sapm_diff r_pre_recent_td_avg  \
0         

In [ ]:
# ============================================================
# ADD EXPONENTIALLY WEIGHTED RECENT-FORM FEATURES
# ============================================================
# This adds recency-weighted fighter stats.
#
# It keeps your existing rolling/career features, but also adds:
# - r_ewm_*
# - b_ewm_*
# - *_ewm_diff
#
# These features give more weight to recent fighter form.
# ============================================================

EWM_SPAN = 3  # lower = more recent-fight emphasis

rolling_df = rolling_df.sort_values("date").reset_index(drop=True)

# Find all red pre-fight rolling stat columns
r_pre_cols = [
    col for col in rolling_df.columns
    if col.startswith("r_pre_")
]

# Match each red stat to its blue version
stat_names = [
    col.replace("r_pre_", "")
    for col in r_pre_cols
    if f"b_pre_{col.replace('r_pre_', '')}" in rolling_df.columns
]

print("Stats to EWM weight:", len(stat_names))
print(stat_names[:20])

Stats to EWM weight: 36
['elo', 'fights', 'wins', 'losses', 'win_pct', 'kd_avg', 'kd_absorbed_avg', 'splm', 'sapm', 'str_acc', 'str_def', 'td_avg', 'td_acc', 'td_def', 'sub_avg', 'ctrl_per_min', 'ctrl_against_per_min', 'finish_rate', 'ko_rate', 'sub_win_rate']


In [ ]:
# ============================================================
# CREATE LONG FIGHTER-LEVEL DATASET
# ============================================================
# Converts fight rows into fighter rows:
#
# One row for red fighter
# One row for blue fighter
#
# This lets us calculate EWM stats by fighter over time.
# ============================================================

fighter_rows = []

for idx, row in rolling_df.iterrows():

    red_row = {
        "fight_index": idx,
        "date": row["date"],
        "fighter_id": row["r_id"],
        "corner": "r"
    }

    blue_row = {
        "fight_index": idx,
        "date": row["date"],
        "fighter_id": row["b_id"],
        "corner": "b"
    }

    for stat in stat_names:
        red_row[stat] = row.get(f"r_pre_{stat}", np.nan)
        blue_row[stat] = row.get(f"b_pre_{stat}", np.nan)

    fighter_rows.append(red_row)
    fighter_rows.append(blue_row)

fighter_long_df = pd.DataFrame(fighter_rows)

fighter_long_df = fighter_long_df.sort_values(
    ["fighter_id", "date"]
).reset_index(drop=True)

fighter_long_df.head()

,fight_index,date,fighter_id,corner,elo,fights,wins,losses,win_pct,kd_avg,...,avg_fight_time,win_streak,loss_streak,days_since_last_fight,recent_win_pct,recent_splm,recent_sapm,recent_td_avg,recent_finish_rate,recent_avg_fight_time
0,8044,2025-06-07,001eb2ab0f30e7ea,b,1500.000000,0,0,0,0.0,0.0,...,0.0,0,0,365,0.0,0.000000,0.000000,0.0,0.0,0.0
1,5131,2019-07-20,002ca196477ce572,b,1500.000000,0,0,0,0.0,0.0,...,0.0,0,0,365,0.0,0.000000,0.000000,0.0,0.0,0.0
2,5436,2020-02-29,002ca196477ce572,r,1484.978655,1,0,1,0.0,0.0,...,300.0,0,1,224,0.0,1.600000,3.000000,1.0,0.0,300.0
3,5426,2020-02-29,003d82fa384ca1d0,r,1500.000000,0,0,0,0.0,0.0,...,0.0,0,0,365,0.0,0.000000,0.000000,0.0,0.0,0.0
4,5901,2021-03-06,003d82fa384ca1d0,b,1484.000000,1,0,1,0.0,0.0,...,85.0,0,1,371,0.0,1.411765,14.117647,0.0,1.0,85.0


In [ ]:
# ============================================================
# CALCULATE EXPONENTIALLY WEIGHTED STATS
# ============================================================
# For each fighter, calculate EWM versions of each stat.
#
# EWM means:
# - newer fights matter more
# - older fights still count, but less
# ============================================================

for stat in stat_names:

    fighter_long_df[f"ewm_{stat}"] = (
        fighter_long_df
        .groupby("fighter_id")[stat]
        .transform(
            lambda x: x.ewm(
                span=EWM_SPAN,
                adjust=False
            ).mean()
        )
    )

fighter_long_df.head()

,fight_index,date,fighter_id,corner,elo,fights,wins,losses,win_pct,kd_avg,...,ewm_avg_fight_time,ewm_win_streak,ewm_loss_streak,ewm_days_since_last_fight,ewm_recent_win_pct,ewm_recent_splm,ewm_recent_sapm,ewm_recent_td_avg,ewm_recent_finish_rate,ewm_recent_avg_fight_time
0,8044,2025-06-07,001eb2ab0f30e7ea,b,1500.000000,0,0,0,0.0,0.0,...,0.0,0.0,0.0,365.0,0.0,0.000000,0.000000,0.0,0.0,0.0
1,5131,2019-07-20,002ca196477ce572,b,1500.000000,0,0,0,0.0,0.0,...,0.0,0.0,0.0,365.0,0.0,0.000000,0.000000,0.0,0.0,0.0
2,5436,2020-02-29,002ca196477ce572,r,1484.978655,1,0,1,0.0,0.0,...,150.0,0.0,0.5,294.5,0.0,0.800000,1.500000,0.5,0.0,150.0
3,5426,2020-02-29,003d82fa384ca1d0,r,1500.000000,0,0,0,0.0,0.0,...,0.0,0.0,0.0,365.0,0.0,0.000000,0.000000,0.0,0.0,0.0
4,5901,2021-03-06,003d82fa384ca1d0,b,1484.000000,1,0,1,0.0,0.0,...,42.5,0.0,0.5,368.0,0.0,0.705882,7.058824,0.0,0.5,42.5


In [ ]:
# ============================================================
# MERGE EWM FEATURES BACK INTO FIGHT ROWS
# ============================================================
# Adds:
#
# r_ewm_*
# b_ewm_*
#
# back into the main rolling dataframe.
# ============================================================

r_ewm = fighter_long_df[
    fighter_long_df["corner"] == "r"
].copy()

b_ewm = fighter_long_df[
    fighter_long_df["corner"] == "b"
].copy()

r_ewm_cols = ["fight_index"] + [
    f"ewm_{stat}" for stat in stat_names
]

b_ewm_cols = ["fight_index"] + [
    f"ewm_{stat}" for stat in stat_names
]

r_ewm = r_ewm[r_ewm_cols].rename(
    columns={
        f"ewm_{stat}": f"r_ewm_{stat}"
        for stat in stat_names
    }
)

b_ewm = b_ewm[b_ewm_cols].rename(
    columns={
        f"ewm_{stat}": f"b_ewm_{stat}"
        for stat in stat_names
    }
)

rolling_df = rolling_df.merge(
    r_ewm,
    left_index=True,
    right_on="fight_index",
    how="left"
).drop(columns=["fight_index"])

rolling_df = rolling_df.merge(
    b_ewm,
    left_index=True,
    right_on="fight_index",
    how="left"
).drop(columns=["fight_index"])

rolling_df.shape

(8190, 309)

In [ ]:
# ============================================================
# CREATE EWM DIFFERENTIAL FEATURES
# ============================================================
# Creates matchup differences:
#
# red EWM stat - blue EWM stat
#
# Example:
# r_ewm_splm - b_ewm_splm = ewm_splm_diff
# ============================================================

for stat in stat_names:

    r_col = f"r_ewm_{stat}"
    b_col = f"b_ewm_{stat}"
    diff_col = f"ewm_{stat}_diff"

    if r_col in rolling_df.columns and b_col in rolling_df.columns:

        rolling_df[diff_col] = (
            rolling_df[r_col] - rolling_df[b_col]
        )

ewm_diff_cols = [
    col for col in rolling_df.columns
    if col.startswith("ewm_") and col.endswith("_diff")
]

print("EWM diff features added:", len(ewm_diff_cols))
print(ewm_diff_cols[:30])

EWM diff features added: 36
['ewm_elo_diff', 'ewm_fights_diff', 'ewm_wins_diff', 'ewm_losses_diff', 'ewm_win_pct_diff', 'ewm_kd_avg_diff', 'ewm_kd_absorbed_avg_diff', 'ewm_splm_diff', 'ewm_sapm_diff', 'ewm_str_acc_diff', 'ewm_str_def_diff', 'ewm_td_avg_diff', 'ewm_td_acc_diff', 'ewm_td_def_diff', 'ewm_sub_avg_diff', 'ewm_ctrl_per_min_diff', 'ewm_ctrl_against_per_min_diff', 'ewm_finish_rate_diff', 'ewm_ko_rate_diff', 'ewm_sub_win_rate_diff', 'ewm_decision_win_rate_diff', 'ewm_finish_loss_rate_diff', 'ewm_decision_loss_rate_diff', 'ewm_avg_opponent_elo_diff', 'ewm_best_win_elo_diff', 'ewm_worst_loss_elo_diff', 'ewm_avg_fight_time_diff', 'ewm_win_streak_diff', 'ewm_loss_streak_diff', 'ewm_days_since_last_fight_diff']


In [ ]:
# ============================================================
# CREATE RECENT-FORM EDGE FEATURES
# ============================================================
# These compare recent weighted form vs career/pre-fight form.
#
# Positive value:
# - fighter is trending better recently
#
# Negative value:
# - fighter is trending worse recently
# ============================================================

for stat in stat_names:

    r_ewm = f"r_ewm_{stat}"
    b_ewm = f"b_ewm_{stat}"

    r_career = f"r_pre_{stat}"
    b_career = f"b_pre_{stat}"

    if r_ewm in rolling_df.columns and r_career in rolling_df.columns:
        rolling_df[f"r_recent_form_{stat}"] = (
            rolling_df[r_ewm] - rolling_df[r_career]
        )

    if b_ewm in rolling_df.columns and b_career in rolling_df.columns:
        rolling_df[f"b_recent_form_{stat}"] = (
            rolling_df[b_ewm] - rolling_df[b_career]
        )

    if (
        f"r_recent_form_{stat}" in rolling_df.columns
        and
        f"b_recent_form_{stat}" in rolling_df.columns
    ):
        rolling_df[f"recent_form_{stat}_diff"] = (
            rolling_df[f"r_recent_form_{stat}"]
            -
            rolling_df[f"b_recent_form_{stat}"]
        )

recent_form_cols = [
    col for col in rolling_df.columns
    if col.startswith("recent_form_")
]

print("Recent form diff features added:", len(recent_form_cols))
print(recent_form_cols[:30])

Recent form diff features added: 36
['recent_form_elo_diff', 'recent_form_fights_diff', 'recent_form_wins_diff', 'recent_form_losses_diff', 'recent_form_win_pct_diff', 'recent_form_kd_avg_diff', 'recent_form_kd_absorbed_avg_diff', 'recent_form_splm_diff', 'recent_form_sapm_diff', 'recent_form_str_acc_diff', 'recent_form_str_def_diff', 'recent_form_td_avg_diff', 'recent_form_td_acc_diff', 'recent_form_td_def_diff', 'recent_form_sub_avg_diff', 'recent_form_ctrl_per_min_diff', 'recent_form_ctrl_against_per_min_diff', 'recent_form_finish_rate_diff', 'recent_form_ko_rate_diff', 'recent_form_sub_win_rate_diff', 'recent_form_decision_win_rate_diff', 'recent_form_finish_loss_rate_diff', 'recent_form_decision_loss_rate_diff', 'recent_form_avg_opponent_elo_diff', 'recent_form_best_win_elo_diff', 'recent_form_worst_loss_elo_diff', 'recent_form_avg_fight_time_diff', 'recent_form_win_streak_diff', 'recent_form_loss_streak_diff', 'recent_form_days_since_last_fight_diff']


/tmp/ipykernel_6059/1342244179.py:36: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  rolling_df[f"recent_form_{stat}_diff"] = (
/tmp/ipykernel_6059/1342244179.py:22: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  rolling_df[f"r_recent_form_{stat}"] = (
/tmp/ipykernel_6059/1342244179.py:27: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame

In [ ]:
# ============================================================
# ADD V5 ENGINEERED FEATURES
# ============================================================

rolling_df = add_v5_engineered_features(rolling_df)

ENGINEERED_FEATURES = get_engineered_feature_list()

feature_registry_df = save_engineered_feature_registry(
    ENGINEERED_FEATURES,
    paths.feature_registry_path
)

print(f"Saved engineered feature registry to:\n{paths.feature_registry_path}")

display(feature_registry_df)

Saved engineered feature registry to:
/content/drive/MyDrive/UFC_AI/ufc_engineered_feature_registry.csv


,feature
0,age_diff
1,height_diff
2,reach_diff
3,weight_diff
4,striking_edge
5,grappling_edge
6,finish_volatility
7,wrestling_pressure_vs_defense
8,reach_striking_combo
9,chin_risk_diff


In [ ]:
# ============================================================
# FINAL SAVE — ENHANCED ROLLING DATASET
# ============================================================

OUTPUT_PATH = paths.rolling_features_path

rolling_df.to_csv(
    OUTPUT_PATH,
    index=False
)

print("Saved enhanced rolling dataset to:")
print(OUTPUT_PATH)

print("Final dataset shape:", rolling_df.shape)


Saved enhanced rolling dataset to:
/content/drive/MyDrive/UFC_AI/UFC_enhanced_rolling_features_EWM.csv
Final dataset shape: (8190, 483)
